In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Set the random seed for reproducibility and define the dataset size
np.random.seed(42)
N = 1000

In [ ]:
# Define pricing factor functions used to compute a synthetic surge multiplier and generate synthetic data

# Increase price during rush hours
def get_peak_factor(hour):
    if 7 <= hour <= 10 or 17 <= hour <= 21:
        return 1.5
    return 1.0
    
# Apply a weather-based multiplier for rain or snow
def get_weather_factor(rain, snow):
    if rain:
        return 1.2
    elif snow:
        return 1.3
    return 1.0

# Map traffic conditions to pricing multipliers
def get_traffic_factor(traffic):
    mapping = {
        'low': 1.0,
        'medium': 1.15,
        'high': 1.3
    }
    return mapping[traffic]

# Base demand grows during busy hours and bad weather
def get_demand_factor(hour, rain, snow):
    base = 1.0
    if 7 <= hour <= 10 or 17 <= hour <= 21:
        base += 0.5
    if rain:
        base += 0.2
    elif snow:

        base += 0.3

    return base 

 # Combine demand and traffic effects, capping the surge factor
def compute_surge(demand_factor, traffic_factor):  
    surge = demand_factor * traffic_factor
    return min(surge, 3.0)

In [ ]:
# Generate synthetic ride features and compute the final fare for each trip

distance = np.random.uniform(1, 20, N)
hour = np.random.randint(0, 24, N)
rain = np.random.choice([0, 1], N, p=[0.7, 0.3])
snow = np.random.choice([0, 1], N, p=[0.6, 0.4])
traffic = np.random.choice(['low', 'medium', 'high'], N, p=[0.5, 0.3, 0.2])

price = []

for i in range(N):
    # Base fare and per-kilometer rate for each ride
    base_fare = 40
    per_km = 12
    fare = base_fare + per_km * distance[i]

    # Compute all pricing multipliers for this ride
    peak_factor = get_peak_factor(hour[i])
    weather_factor = get_weather_factor(rain[i], snow[i])
    traffic_factor = get_traffic_factor(traffic[i])
    demand_factor = get_demand_factor(hour[i], rain[i], snow[i])
    surge = compute_surge(demand_factor, traffic_factor)
    
    # Apply surge pricing and additional multipliers
    fare = fare * peak_factor * weather_factor * surge

    # Add random noise to simulate real-world fare variation
    noise = np.random.normal(0, 8)    
    fare += noise

    price.append(round(fare, 2))

In [ ]:
# Assemble the synthetic dataset into a DataFrame and save it to disk

df = pd.DataFrame({
    'distance': distance,
    'hour': hour,
    'rain': rain,
    'snow': snow,
    'traffic': traffic,
    'price': price
})

df.to_csv('synthetic_rides.csv', index=False)
print("Dataset created and saved as synthetic_rides.csv")

Dataset created and saved as synthetic_rides.csv


In [8]:
# Load the generated dataset and perform a quick quality check

print("Dataset shape:\n", df.shape)
print("\nFirst 5 rows:\n", df.head())

print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:\n", df.duplicated().sum())

print("\nTraffic distribution:\n", df["traffic"].value_counts())
print("\nRain distribution:\n", df["rain"].value_counts())

Dataset shape:
 (1000, 6)

First 5 rows:
     distance  hour  rain  snow traffic       price
0   8.116262    14     0     1  medium  260.145927
1  19.063572    11     1     0     low  405.376670
2  14.907885    15     0     1     low  372.962390
3  12.374511    23     1     1  medium  315.168920
4   3.964354    18     1     0     low  263.226508

Missing values:
 distance    0
hour        0
rain        0
snow        0
traffic     0
price       0
dtype: int64

Duplicate rows:
 0

Traffic distribution:
 traffic
low       503
medium    302
high      195
Name: count, dtype: int64

Rain distribution:
 rain
0    699
1    301
Name: count, dtype: int64
